## インプット

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import paramiko
from scp import SCPClient
import os
import pandas as pd
import numpy as np
import re
import time
import subprocess
import ast
from datetime import datetime
import shutil
from datetime import datetime

from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split,KFold, cross_validate,cross_val_predict
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error,make_scorer
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.data import Data, DataLoader
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.metrics import log_loss
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from itertools import product
import gc
import random
from torch.utils.data import Dataset, DataLoader
import copy
import math
import posixpath

### サーバーに接続

In [24]:
def connect_server():
    try:
        client = paramiko.SSHClient()
        client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        # あなたのサーバー情報に合わせて書き換えてください
        client.connect('192.168.11.20', username='teraimao', key_filename='/Users/teraimao/.ssh/id_ed25519')
        return client
    except Exception as e:
        print(f"Connection failed: {e}")
        return None

# 関数を呼び出して、戻り値を 'client' という変数に入れる
client = connect_server()

if client:
    print("✅ サーバーへの接続に成功しました。'client' が定義されました。")
else:
    print("❌ 接続に失敗しました。設定を確認してください。")

✅ サーバーへの接続に成功しました。'client' が定義されました。


## ほんちゃん

In [25]:
# =========================================================
# 安全版 MD解析データ集計 helper
# GBSW / HDGB / HDGBvdW 対応
# model と 保存CSV名の食い違いを防ぐ版
# =========================================================

SAVE_DIR = "/Users/teraimao/experiment/data/解析結果/"
os.makedirs(SAVE_DIR, exist_ok=True)

REMOTE_SCRIPT_PATH = "/home/teraimao/experiment/scripts/aggregate_md_fast_universal.py"
REMOTE_OUT_DIR = "/home/teraimao/experiment/aggregation_results"

# =========================================================
# 固定モデル設定
# "gbsw", "hdgb", "hdgbvdw" のどれか
# =========================================================

FIXED_MODEL = "hdgb"


def validate_model(model):
    """
    model名を正規化・検査する。
    """
    model = str(model).lower().strip()

    if model not in ["gbsw", "hdgb", "hdgbvdw"]:
        raise ValueError("model は 'gbsw', 'hdgb', 'hdgbvdw' のどれかにしてください")

    return model

def normalize_csv_name(name, model=None):
    """
    CSV名を整える。
    modelを明示して、FIXED_MODELへの依存を減らす。
    """
    model = validate_model(model or FIXED_MODEL)

    name = str(name).strip()

    if name == "":
        name = f"master_database_{model}.csv"

    # パスが入っていた場合はファイル名だけ使う
    name = os.path.basename(name)

    if not name.endswith(".csv"):
        name += ".csv"

    return name


def check_model_csv_consistency(model, save_csv_name):
    """
    modelと保存CSV名の食い違いを防ぐ。

    例：
      model=hdgb なのに master_database_hdgbvdw.csv はNG
      model=hdgbvdw なのに master_database_hdgb.csv はNG
    """
    model = validate_model(model)
    save_csv_name = normalize_csv_name(save_csv_name, model=model)

    lower_name = save_csv_name.lower()
    expected = f"master_database_{model}"

    if not lower_name.startswith(expected):
        raise ValueError(
            "model と 保存CSV名 が食い違っています。\n"
            f"model        : {model}\n"
            f"save_csv_name: {save_csv_name}\n"
            f"期待されるCSV名の例: master_database_{model}.csv\n"
            "このままだと hdgb と hdgbvdw が混ざる可能性があります。"
        )

    return save_csv_name

def get_custom_master_path(csv_name, model=None):
    """
    ローカルmaster CSVの保存先パスを返す。
    """
    model = validate_model(model or FIXED_MODEL)
    csv_name = normalize_csv_name(csv_name, model=model)
    return os.path.join(SAVE_DIR, csv_name)

# =========================================================
# model と work親ディレクトリの整合性チェック
# =========================================================

MODEL_REMOTE_DIR_CONFIG = {
    "gbsw": {
        "subdir_prefix": "gbsw",
        "analysis_prefix": "ana_gbsw",
    },
    "hdgb": {
        "subdir_prefix": "hdgb",
        "analysis_prefix": "ana_hdgb",
    },
    "hdgbvdw": {
        "subdir_prefix": "hdgbvdw",
        "analysis_prefix": "ana_hdgbvdw",
    },
}



def check_remote_model_dirs(client, model, base_dir, start_id, end_id):
    """
    サーバー上で、選択したmodelに対応する
      model_dir: hdgb_1 / hdgbvdw_1 / gbsw_1
      analysis_dir: ana_hdgb_1 / ana_hdgbvdw_1 / ana_gbsw_1
    が実際に存在するか確認する。

    範囲内で最初に見つかったworkを使ってチェックする。
    """
    model = validate_model(model)
    cfg = MODEL_REMOTE_DIR_CONFIG[model]

    subdir_prefix = cfg["subdir_prefix"]
    analysis_prefix = cfg["analysis_prefix"]

    start_id = int(start_id)
    end_id = int(end_id)

    cmd = f"""
base='{base_dir}'
start_id={start_id}
end_id={end_id}
subdir_prefix='{subdir_prefix}'
analysis_prefix='{analysis_prefix}'

echo '===== MODEL/BASE CHECK ====='
echo "model={model}"
echo "base=$base"
echo "subdir_prefix=$subdir_prefix"
echo "analysis_prefix=$analysis_prefix"

if [ ! -d "$base" ]; then
    echo "__BASE_NOT_FOUND__"
    exit 0
fi

found_work=""

for i in $(seq "$start_id" "$end_id"); do
    w=$(find "$base" -maxdepth 1 -type d -name "${{i}}_*work" 2>/dev/null | head -n 1)
    if [ -n "$w" ]; then
        found_work="$w"
        break
    fi
done

if [ -z "$found_work" ]; then
    echo "__NO_WORK_IN_RANGE__"
    exit 0
fi

echo "found_work=$found_work"

if [ -d "$found_work/${{subdir_prefix}}_1" ]; then
    echo "__MD_DIR_OK__"
else
    echo "__MD_DIR_NG__"
    echo "missing: $found_work/${{subdir_prefix}}_1"
fi

if [ -d "$found_work/analysis/${{analysis_prefix}}_1" ]; then
    echo "__ANALYSIS_DIR_OK__"
else
    echo "__ANALYSIS_DIR_NG__"
    echo "missing: $found_work/analysis/${{analysis_prefix}}_1"
fi
"""

    stdin, stdout, stderr = client.exec_command(cmd)
    out = stdout.read().decode()
    err = stderr.read().decode()

    print(out)

    if err:
        print("stderr:")
        print(err)

    if "__BASE_NOT_FOUND__" in out:
        raise ValueError(
            "指定したwork親ディレクトリがサーバー上に存在しません。\n"
            f"base_dir: {base_dir}"
        )

    if "__NO_WORK_IN_RANGE__" in out:
        raise ValueError(
            "指定ID範囲内にworkディレクトリが見つかりません。\n"
            f"base_dir: {base_dir}\n"
            f"ID範囲: {start_id} ~ {end_id}"
        )

    if "__MD_DIR_NG__" in out:
        raise ValueError(
            "選択したmodelに対応する計算ディレクトリが見つかりません。\n"
            f"model: {model}\n"
            f"期待: {subdir_prefix}_1\n"
            "model と work親ディレクトリの組み合わせがズレている可能性が高いです。"
        )

    if "__ANALYSIS_DIR_NG__" in out:
        raise ValueError(
            "選択したmodelに対応するanalysisディレクトリが見つかりません。\n"
            f"model: {model}\n"
            f"期待: analysis/{analysis_prefix}_1\n"
            "model と work親ディレクトリの組み合わせを確認してください。"
        )

    return True

def submit_md_aggregation_background_custom_name(
    client,
    model,
    base_dir,
    start_id,
    end_id,
    target_date="ALL",
    target_t_end_ps="500",
    skip_ps=250.0,
    dt_ps=1.0,
    tolerance_frames=0,
    save_csv_name=None,
):
    """
    サーバー側で集計をバックグラウンド実行。
    ローカル保存用CSV名を指定できる版。

    安全版：
      - model と save_csv_name の食い違いを防ぐ
      - base_dir名には依存しない
      - modelに応じて、work内の subdir / analysis dir だけ確認する
    """

    # 注意：
    # ここでは install_remote_universal_aggregator は呼ばない。
    # 古いスクリプトで上書きしてしまう可能性があるため。

    model = validate_model(model)

    # -----------------------------------------------------
    # 1. サーバー上で実際のwork構造を確認
    # base_dir名ではなく、work内の
    #   gbsw_1 / hdgb_1 / hdgbvdw_1
    #   analysis/ana_gbsw_1 / ana_hdgb_1 / ana_hdgbvdw_1
    # を見る
    # -----------------------------------------------------

    check_remote_model_dirs(
        client=client,
        model=model,
        base_dir=base_dir,
        start_id=start_id,
        end_id=end_id
    )

    # -----------------------------------------------------
    # 2. 保存CSV名の整合性チェック
    # -----------------------------------------------------

    save_csv_name = normalize_csv_name(
        save_csv_name or f"master_database_{model}.csv",
        model=model
    )

    save_csv_name = check_model_csv_consistency(
        model,
        save_csv_name
    )

    base_name = save_csv_name.replace(".csv", "")

    tag_date = str(target_date).replace("*", "ALL").replace("/", "_")
    tag_tend = str(target_t_end_ps).replace("*", "ALL")
    timestamp = time.strftime("%Y%m%d_%H%M%S")

    remote_csv = (
        f"{REMOTE_OUT_DIR}/{base_name}_"
        f"{model}_ID{start_id}_{end_id}_{tag_date}_{tag_tend}ps_{timestamp}.csv"
    )

    remote_log = remote_csv.replace(".csv", ".log")

    cmd = (
        f"nohup python3 {REMOTE_SCRIPT_PATH} "
        f"--model '{model}' "
        f"--base_dir '{base_dir}' "
        f"--start {start_id} "
        f"--end {end_id} "
        f"--date '{target_date}' "
        f"--tend '{target_t_end_ps}' "
        f"--skip {skip_ps} "
        f"--dt {dt_ps} "
        f"--tolerance_frames {tolerance_frames} "
        f"--out '{remote_csv}' "
        f"> '{remote_log}' 2>&1 & echo $!"
    )

    stdin, stdout, stderr = client.exec_command(cmd)
    pid = stdout.read().decode().strip()
    err = stderr.read().decode().strip()

    if err:
        print("stderr:", err)

    print("🚀 集計をバックグラウンド投入しました")
    print("model:", model)
    print("base_dir:", base_dir)
    print("PID:", pid)
    print("remote_csv:", remote_csv)
    print("remote_log:", remote_log)
    print("local_save_csv:", save_csv_name)

    if f"_{model}_ID" not in remote_csv:
        raise RuntimeError(
            "remote_csv名にmodel名が正しく入っていません。\n"
            f"model: {model}\n"
            f"remote_csv: {remote_csv}"
        )

    return {
        "model": model,
        "pid": pid,
        "remote_csv": remote_csv,
        "remote_log": remote_log,
        "base_dir": base_dir,
        "save_csv_name": save_csv_name,
    }

def download_and_merge_md_aggregation_custom_name(client, job):
    """
    完了CSVをダウンロードし、指定したCSV名に統合する。
    """

    model = validate_model(job["model"])
    remote_csv = job["remote_csv"]

    save_csv_name = normalize_csv_name(
        job.get("save_csv_name", f"master_database_{model}.csv"),
        model=model
    )

    save_csv_name = check_model_csv_consistency(model, save_csv_name)

    local_master_csv = get_custom_master_path(save_csv_name, model=model)

    local_tmp = os.path.join(
        SAVE_DIR,
        os.path.basename(remote_csv)
    )

    sftp = client.open_sftp()
    try:
        sftp.get(remote_csv, local_tmp)
    finally:
        sftp.close()

    print("✅ downloaded temporary csv:")
    print(local_tmp)

    new_df = pd.read_csv(local_tmp)

    # 念のため、CSV内のModel列も確認
    if "Model" in new_df.columns:
        actual_models = set(
            new_df["Model"].dropna().astype(str).str.lower().unique()
        )

        if actual_models and actual_models != {model}:
            raise ValueError(
                "ダウンロードしたCSV内のModel列が想定と違います。\n"
                f"expected model: {model}\n"
                f"actual models : {actual_models}\n"
                "このCSVは統合しない方が安全です。"
            )

    if os.path.exists(local_master_csv):
        old_df = pd.read_csv(local_master_csv)

        final_df = pd.concat([old_df, new_df], ignore_index=True)

        final_df = final_df.drop_duplicates(
            subset=["Model", "ID", "Date", "Pos_Num", "t_end_ps"],
            keep="last"
        )
    else:
        final_df = new_df

    # 数値列を数値化してからsort
    for col in ["ID", "Pos_Num", "t_end_ps"]:
        if col in final_df.columns:
            final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

    final_df = final_df.sort_values(
        ["ID", "Date", "t_end_ps", "Pos_Num"]
    ).reset_index(drop=True)

    final_df.to_csv(local_master_csv, index=False)

    print("✅ merged master saved:")
    print(local_master_csv)
    print("master shape:", final_df.shape)

    if "Model" in final_df.columns:
        print("\nModel counts:")
        print(final_df["Model"].value_counts(dropna=False))

    if "Status" in final_df.columns:
        print("\nStatus counts:")
        print(final_df["Status"].value_counts(dropna=False))

    display(final_df.tail(10))

    return final_df


def check_md_aggregation_job(client, job):
    """
    バックグラウンド集計ジョブの進捗確認。
    """
    pid = job.get("pid")
    remote_csv = job["remote_csv"]
    remote_log = job["remote_log"]

    if pid is None or str(pid).strip() == "":
        pid_check = "echo 'PIDなし'"
    else:
        pid_check = f"ps -p {pid} -o pid,stat,etime,cmd"

    cmd = f"""
echo '===== PID CHECK ====='
{pid_check}

echo ''
echo '===== CSV CHECK ====='
ls -lh '{remote_csv}' 2>/dev/null || echo 'CSV not created yet'

echo ''
echo '===== LOG CHECK ====='
ls -lh '{remote_log}' 2>/dev/null || echo 'LOG not created yet'

echo ''
echo '===== LOG TAIL ====='
tail -n 80 '{remote_log}' 2>/dev/null || echo 'No log yet'

echo ''
echo '===== CSV HEAD ====='
head -n 3 '{remote_csv}' 2>/dev/null || echo 'No csv yet'

echo ''
echo '===== CSV LINE COUNT ====='
wc -l '{remote_csv}' 2>/dev/null || echo 'No csv yet'
"""

    stdin, stdout, stderr = client.exec_command(cmd)

    out = stdout.read().decode()
    err = stderr.read().decode()

    print(out)

    if err:
        print("stderr:")
        print(err)


print("✅ safe MD aggregation helper loaded")
print("FIXED_MODEL:", FIXED_MODEL)
print("default save csv:", normalize_csv_name("", model=FIXED_MODEL))


✅ safe MD aggregation helper loaded
FIXED_MODEL: hdgb
default save csv: master_database_hdgb.csv


In [26]:

# =========================================================
# modelに応じて、work内で探すディレクトリ名を決める
# base_dirの名前には依存しない版
# =========================================================

MODEL_REMOTE_DIR_CONFIG = {
    "gbsw": {
        "subdir_prefix": "gbsw",
        "analysis_prefix": "ana_gbsw",
    },
    "hdgb": {
        "subdir_prefix": "hdgb",
        "analysis_prefix": "ana_hdgb",
    },
    "hdgbvdw": {
        "subdir_prefix": "hdgbvdw",
        "analysis_prefix": "ana_hdgbvdw",
    },
}


def check_remote_model_dirs(client, model, base_dir, start_id, end_id):
    """
    base_dirの名前ではなく、
    選択したmodelに対応するwork内ディレクトリがあるかだけ確認する。

    gbsw:
      gbsw_1
      analysis/ana_gbsw_1

    hdgb:
      hdgb_1
      analysis/ana_hdgb_1

    hdgbvdw:
      hdgbvdw_1
      analysis/ana_hdgbvdw_1

    ※ hdgbvdw work内に analysis/ana_hdgb_1 があっても無視する。
    """

    model = validate_model(model)
    cfg = MODEL_REMOTE_DIR_CONFIG[model]

    subdir_prefix = cfg["subdir_prefix"]
    analysis_prefix = cfg["analysis_prefix"]

    start_id = int(start_id)
    end_id = int(end_id)

    cmd = f"""
base='{base_dir}'
start_id={start_id}
end_id={end_id}
subdir_prefix='{subdir_prefix}'
analysis_prefix='{analysis_prefix}'

echo '===== MODEL/WORK STRUCTURE CHECK ====='
echo "model=$'{model}'"
echo "base=$base"
echo "expected md dir: ${{subdir_prefix}}_1"
echo "expected analysis dir: analysis/${{analysis_prefix}}_1"

if [ ! -d "$base" ]; then
    echo "__BASE_NOT_FOUND__"
    exit 0
fi

found_work=""

for i in $(seq "$start_id" "$end_id"); do
    w=$(find "$base" -maxdepth 1 -type d -name "${{i}}_*work" 2>/dev/null | head -n 1)
    if [ -n "$w" ]; then
        found_work="$w"
        break
    fi
done

if [ -z "$found_work" ]; then
    echo "__NO_WORK_IN_RANGE__"
    exit 0
fi

echo "found_work=$found_work"

if [ -d "$found_work/${{subdir_prefix}}_1" ]; then
    echo "__MD_DIR_OK__"
else
    echo "__MD_DIR_NG__"
    echo "missing: $found_work/${{subdir_prefix}}_1"
fi

if [ -d "$found_work/analysis/${{analysis_prefix}}_1" ]; then
    echo "__ANALYSIS_DIR_OK__"
else
    echo "__ANALYSIS_DIR_NG__"
    echo "missing: $found_work/analysis/${{analysis_prefix}}_1"
fi
"""

    stdin, stdout, stderr = client.exec_command(cmd)
    out = stdout.read().decode()
    err = stderr.read().decode()

    print(out)

    if err:
        print("stderr:")
        print(err)

    if "__BASE_NOT_FOUND__" in out:
        raise ValueError(
            "指定したwork親ディレクトリがサーバー上に存在しません。\n"
            f"base_dir: {base_dir}"
        )

    if "__NO_WORK_IN_RANGE__" in out:
        raise ValueError(
            "指定ID範囲内にworkディレクトリが見つかりません。\n"
            f"base_dir: {base_dir}\n"
            f"ID範囲: {start_id} ~ {end_id}"
        )

    if "__MD_DIR_NG__" in out:
        raise ValueError(
            "選択したmodelに対応する計算ディレクトリが見つかりません。\n"
            f"model: {model}\n"
            f"期待: {subdir_prefix}_1\n"
            "base_dir内のworkが、選択したmodelの計算結果ではない可能性があります。"
        )

    if "__ANALYSIS_DIR_NG__" in out:
        raise ValueError(
            "選択したmodelに対応するanalysisディレクトリが見つかりません。\n"
            f"model: {model}\n"
            f"期待: analysis/{analysis_prefix}_1\n"
            "解析ディレクトリ名を確認してください。"
        )

    return True


def submit_md_aggregation_background_custom_name(
    client,
    model,
    base_dir,
    start_id,
    end_id,
    target_date="ALL",
    target_t_end_ps="500",
    skip_ps=250.0,
    dt_ps=1.0,
    tolerance_frames=0,
    save_csv_name=None,
):
    """
    サーバー側で集計をバックグラウンド実行。
    ローカル保存用CSV名を指定できる版。

    安全版：
      - model と save_csv_name の食い違いを防ぐ
      - base_dir名には依存しない
      - modelに応じて、work内の subdir / analysis dir だけ確認する
    """

    # 注意：
    # ここでは install_remote_universal_aggregator は呼ばない。
    # 古いスクリプトで上書きしてしまう可能性があるため。

    model = validate_model(model)

    # -----------------------------------------------------
    # 1. サーバー上で実際のwork構造を確認
    # base_dir名ではなく、work内の
    #   gbsw_1 / hdgb_1 / hdgbvdw_1
    #   analysis/ana_gbsw_1 / ana_hdgb_1 / ana_hdgbvdw_1
    # を見る
    # -----------------------------------------------------

    check_remote_model_dirs(
        client=client,
        model=model,
        base_dir=base_dir,
        start_id=start_id,
        end_id=end_id
    )

    # -----------------------------------------------------
    # 2. 保存CSV名の整合性チェック
    # -----------------------------------------------------

    save_csv_name = normalize_csv_name(
        save_csv_name or f"master_database_{model}.csv",
        model=model
    )

    save_csv_name = check_model_csv_consistency(
        model,
        save_csv_name
    )

    base_name = save_csv_name.replace(".csv", "")

    tag_date = str(target_date).replace("*", "ALL").replace("/", "_")
    tag_tend = str(target_t_end_ps).replace("*", "ALL")
    timestamp = time.strftime("%Y%m%d_%H%M%S")

    remote_csv = (
        f"{REMOTE_OUT_DIR}/{base_name}_"
        f"{model}_ID{start_id}_{end_id}_{tag_date}_{tag_tend}ps_{timestamp}.csv"
    )

    remote_log = remote_csv.replace(".csv", ".log")

    cmd = (
        f"nohup python3 {REMOTE_SCRIPT_PATH} "
        f"--model '{model}' "
        f"--base_dir '{base_dir}' "
        f"--start {start_id} "
        f"--end {end_id} "
        f"--date '{target_date}' "
        f"--tend '{target_t_end_ps}' "
        f"--skip {skip_ps} "
        f"--dt {dt_ps} "
        f"--tolerance_frames {tolerance_frames} "
        f"--out '{remote_csv}' "
        f"> '{remote_log}' 2>&1 & echo $!"
    )

    stdin, stdout, stderr = client.exec_command(cmd)
    pid = stdout.read().decode().strip()
    err = stderr.read().decode().strip()

    if err:
        print("stderr:", err)

    print("🚀 集計をバックグラウンド投入しました")
    print("model:", model)
    print("base_dir:", base_dir)
    print("PID:", pid)
    print("remote_csv:", remote_csv)
    print("remote_log:", remote_log)
    print("local_save_csv:", save_csv_name)

    if f"_{model}_ID" not in remote_csv:
        raise RuntimeError(
            "remote_csv名にmodel名が正しく入っていません。\n"
            f"model: {model}\n"
            f"remote_csv: {remote_csv}"
        )

    return {
        "model": model,
        "pid": pid,
        "remote_csv": remote_csv,
        "remote_log": remote_log,
        "base_dir": base_dir,
        "save_csv_name": save_csv_name,
    }


In [27]:
# =========================================================
# モデル選択版 UI
# 安全版 helper 対応
# GBSW / HDGB / HDGBvdW をUIで選択できる版
# =========================================================


current_job = {"job": None}

style = {"description_width": "initial"}


# =========================================================
# モデルごとのデフォルトwork親ディレクトリ
# 必要ならUI上で変更してOK
# =========================================================

DEFAULT_BASE_DIR_BY_MODEL = {
    "gbsw": "/home/teraimao/experiment/confirm",
    "hdgb": "/home/teraimao/experiment/confirm_hdgb_500ps_z軸0,7,15,17,20",
    "hdgbvdw": "/home/teraimao/experiment/confirm_hdgbvdw_500ps_一定_z軸0,7,15,17,20",
}


def default_base_dir_for_model(model):
    model = validate_model(model)
    return DEFAULT_BASE_DIR_BY_MODEL.get(
        model,
        "/home/teraimao/experiment/confirm"
    )


def default_save_csv_for_model(model):
    model = validate_model(model)
    return normalize_csv_name(
        f"master_database_{model}.csv",
        model=model
    )


# =========================================================
# UI部品
# =========================================================

model_box = widgets.Dropdown(
    options=[
        ("GBSW", "gbsw"),
        ("HDGB", "hdgb"),
        ("HDGBvdW", "hdgbvdw"),
    ],
    value=validate_model(globals().get("FIXED_MODEL", "hdgb")),
    description="モデル:",
    style=style,
    layout=widgets.Layout(width="260px")
)

txt_base_dir = widgets.Text(
    value=default_base_dir_for_model(model_box.value),
    description="work親ディレクトリ:",
    style=style,
    layout=widgets.Layout(width="760px")
)

txt_save_csv = widgets.Text(
    value=default_save_csv_for_model(model_box.value),
    description="保存CSV名:",
    style=style,
    layout=widgets.Layout(width="520px")
)

txt_start = widgets.IntText(
    value=1,
    description="開始ID:",
    style=style,
    layout=widgets.Layout(width="160px")
)

txt_end = widgets.IntText(
    value=100,
    description="終了ID:",
    style=style,
    layout=widgets.Layout(width="160px")
)

txt_date = widgets.Text(
    value="ALL",
    description="日付 (ALLで全日程):",
    style=style,
    layout=widgets.Layout(width="260px")
)

txt_tend = widgets.Text(
    value="500",
    description="計算時間ps (ALLで全件):",
    style=style,
    layout=widgets.Layout(width="260px")
)

txt_skip = widgets.FloatText(
    value=250.0,
    description="skip_ps:",
    style=style,
    layout=widgets.Layout(width="180px")
)

txt_dt = widgets.FloatText(
    value=1.0,
    description="dt_ps:",
    style=style,
    layout=widgets.Layout(width="160px")
)

txt_tol = widgets.IntText(
    value=0,
    description="許容欠損frame:",
    style=style,
    layout=widgets.Layout(width="200px")
)

btn_submit = widgets.Button(
    description="🚀 集計をバックグラウンド開始",
    button_style="danger",
    layout=widgets.Layout(width="300px", height="42px")
)

btn_check = widgets.Button(
    description="📡 進捗確認",
    button_style="info",
    layout=widgets.Layout(width="160px", height="42px")
)

btn_download = widgets.Button(
    description="📥 完了CSVを取得して指定CSVに統合",
    button_style="success",
    layout=widgets.Layout(width="340px", height="42px")
)

btn_show_master = widgets.Button(
    description="📄 保存CSV確認",
    button_style="",
    layout=widgets.Layout(width="180px", height="42px")
)

out_log = widgets.Output()


# =========================================================
# モデル変更時にwork親dirと保存CSV名を自動更新
# =========================================================

def on_model_changed(change):
    if change["name"] != "value":
        return

    model = validate_model(change["new"])

    # work親ディレクトリは自動変更しない
    # ディレクトリ名は毎回変わるので、ユーザーが手入力した値を残す
    txt_save_csv.value = default_save_csv_for_model(model)

    current_job["job"] = None

    with out_log:
        clear_output()
        print("✅ モデルを変更しました")
        print("model:", model)
        print("work親ディレクトリは自動変更していません。必要なら手入力してください。")
        print("現在のwork親ディレクトリ:", txt_base_dir.value)
        print("保存CSV名:", txt_save_csv.value)
        print("※ 前のjob情報はリセットしました。")


# =========================================================
# ボタン動作
# =========================================================

def on_submit_clicked(b):
    with out_log:
        clear_output()

        try:
            model = validate_model(model_box.value)

            # グローバルにも反映しておく
            globals()["FIXED_MODEL"] = model

            save_csv_name = normalize_csv_name(
                txt_save_csv.value,
                model=model
            )

            save_csv_name = check_model_csv_consistency(
                model,
                save_csv_name
            )

            local_save_path = get_custom_master_path(
                save_csv_name,
                model=model
            )

            print("🚀 MD解析データ集計をバックグラウンドで開始します")
            print(f"モデル        : {model}")
            print(f"work親dir     : {txt_base_dir.value}")
            print(f"保存CSV名     : {save_csv_name}")
            print(f"ローカル保存先 : {local_save_path}")
            print(f"ID範囲        : {txt_start.value} ~ {txt_end.value}")
            print(f"日付          : {txt_date.value}")
            print(f"計算時間ps    : {txt_tend.value}")
            print(f"skip_ps       : {txt_skip.value}")
            print(f"dt_ps         : {txt_dt.value}")
            print(f"許容欠損      : {txt_tol.value}")
            print("-" * 90)

            job = submit_md_aggregation_background_custom_name(
                client=client,
                model=model,
                base_dir=txt_base_dir.value,
                start_id=txt_start.value,
                end_id=txt_end.value,
                target_date=txt_date.value,
                target_t_end_ps=txt_tend.value,
                skip_ps=txt_skip.value,
                dt_ps=txt_dt.value,
                tolerance_frames=txt_tol.value,
                save_csv_name=save_csv_name,
            )

            current_job["job"] = job

            print("\n✅ 投入完了。Notebookはこのまま他のセルを実行できます。")
            print("\n===== JOB INFO =====")
            print("model      :", job["model"])
            print("PID        :", job["pid"])
            print("remote_csv :", job["remote_csv"])
            print("remote_log :", job["remote_log"])
            print("save_csv   :", job["save_csv_name"])

            if f"_{model}_ID" in job["remote_csv"]:
                print("\n✅ remote_csv のmodel名チェックOK")
            else:
                print("\n⚠️ remote_csv のmodel名が怪しいです。確認してください。")

        except Exception as e:
            print("❌ エラー:", e)


def on_check_clicked(b):
    with out_log:
        clear_output()

        job = current_job.get("job")

        if job is None:
            print("⚠️ まだjobが投入されていません。")
            return

        print("📡 進捗確認中...")
        print("-" * 90)

        try:
            check_md_aggregation_job(client, job)
        except Exception as e:
            print("❌ エラー:", e)


def on_download_clicked(b):
    with out_log:
        clear_output()

        job = current_job.get("job")

        if job is None:
            print("⚠️ まだjobが投入されていません。")
            return

        print("📥 CSVをダウンロードして指定CSVに統合します")
        print("-" * 90)

        try:
            df_master = download_and_merge_md_aggregation_custom_name(
                client=client,
                job=job,
            )

            print("\n✅ 統合完了")
            print("shape:", df_master.shape)

        except Exception as e:
            print("❌ エラー:", e)


def on_show_master_clicked(b):
    with out_log:
        clear_output()

        try:
            model = validate_model(model_box.value)

            save_csv_name = normalize_csv_name(
                txt_save_csv.value,
                model=model
            )

            save_csv_name = check_model_csv_consistency(
                model,
                save_csv_name
            )

            master_path = get_custom_master_path(
                save_csv_name,
                model=model
            )

            if not os.path.exists(master_path):
                print("⚠️ 指定CSVがまだありません。")
                print(master_path)
                return

            df = pd.read_csv(master_path)

            print("📄 保存CSV確認")
            print("path :", master_path)
            print("shape:", df.shape)

            if "Model" in df.columns:
                print("\nModel counts:")
                print(df["Model"].value_counts(dropna=False))

            if "Status" in df.columns:
                print("\nStatus counts:")
                print(df["Status"].value_counts(dropna=False))

            if "ID" in df.columns:
                print("\nID range:")
                print(df["ID"].min(), "~", df["ID"].max())

            if "Date" in df.columns:
                print("\nDate counts:")
                print(df["Date"].value_counts().head())

            if "t_end_ps" in df.columns:
                print("\nt_end_ps counts:")
                print(df["t_end_ps"].value_counts().head())

            if "LogPath" in df.columns:
                print("\nLogPath check:")
                print("hdgbvdw in LogPath:", df["LogPath"].astype(str).str.contains("hdgbvdw").sum())
                print("hdgb_ in LogPath:", df["LogPath"].astype(str).str.contains("/hdgb_").sum())
                print("gbsw_ in LogPath:", df["LogPath"].astype(str).str.contains("/gbsw_").sum())

            display(df.tail(10))

        except Exception as e:
            print("❌ エラー:", e)


# =========================================================
# ボタン接続
# =========================================================

btn_submit.on_click(on_submit_clicked)
btn_check.on_click(on_check_clicked)
btn_download.on_click(on_download_clicked)
btn_show_master.on_click(on_show_master_clicked)


# =========================================================
# UI表示
# =========================================================

ui = widgets.VBox([
    widgets.HTML("<h3>🐚 MD解析データ集計：モデル選択・保存CSV名指定版</h3>"),
    widgets.HTML(
        "<p>モデルをプルダウンで選択できます。"
        "保存CSV名とwork親ディレクトリはモデル変更時に自動更新されます。</p>"
    ),
    widgets.HBox([model_box]),
    widgets.HBox([txt_base_dir]),
    widgets.HBox([txt_save_csv]),
    widgets.HBox([txt_start, txt_end, txt_date, txt_tend]),
    widgets.HBox([txt_skip, txt_dt, txt_tol]),
    widgets.HBox([btn_submit, btn_check]),
    widgets.HBox([btn_download, btn_show_master]),
    out_log
])

display(ui)

print("✅ model-select aggregation UI loaded")
print("selected model:", model_box.value)
print("default work親dir:", txt_base_dir.value)
print("default save csv:", txt_save_csv.value)


✅ model-select aggregation UI loaded
selected model: hdgb
default work親dir: /home/teraimao/experiment/confirm_hdgb_500ps_z軸0,7,15,17,20
default save csv: master_database_hdgb.csv


### 終わり

## ほんちゃん２

In [6]:
# =========================================================
# 安全版 MD解析データ集計 helper
# GBSW / HDGB / HDGBvdW 対応
# model と 保存CSV名の食い違いを防ぐ版
# =========================================================

SAVE_DIR = "/Users/teraimao/experiment/data/解析結果/"
os.makedirs(SAVE_DIR, exist_ok=True)

REMOTE_SCRIPT_PATH = "/home/teraimao/experiment/scripts/aggregate_md_fast_universal.py"
REMOTE_OUT_DIR = "/home/teraimao/experiment/aggregation_results"

# =========================================================
# 固定モデル設定
# "gbsw", "hdgb", "hdgbvdw" のどれか
# =========================================================

FIXED_MODEL = "hdgb"


def validate_model(model):
    """
    model名を正規化・検査する。
    """
    model = str(model).lower().strip()

    if model not in ["gbsw", "hdgb", "hdgbvdw"]:
        raise ValueError("model は 'gbsw', 'hdgb', 'hdgbvdw' のどれかにしてください")

    return model

def normalize_csv_name(name, model=None):
    """
    CSV名を整える。
    modelを明示して、FIXED_MODELへの依存を減らす。
    """
    model = validate_model(model or FIXED_MODEL)

    name = str(name).strip()

    if name == "":
        name = f"master_database_{model}.csv"

    # パスが入っていた場合はファイル名だけ使う
    name = os.path.basename(name)

    if not name.endswith(".csv"):
        name += ".csv"

    return name


def check_model_csv_consistency(model, save_csv_name):
    """
    modelと保存CSV名の食い違いを防ぐ。

    例：
      model=hdgb なのに master_database_hdgbvdw.csv はNG
      model=hdgbvdw なのに master_database_hdgb.csv はNG
    """
    model = validate_model(model)
    save_csv_name = normalize_csv_name(save_csv_name, model=model)

    lower_name = save_csv_name.lower()
    expected = f"master_database_{model}"

    if not lower_name.startswith(expected):
        raise ValueError(
            "model と 保存CSV名 が食い違っています。\n"
            f"model        : {model}\n"
            f"save_csv_name: {save_csv_name}\n"
            f"期待されるCSV名の例: master_database_{model}.csv\n"
            "このままだと hdgb と hdgbvdw が混ざる可能性があります。"
        )

    return save_csv_name

def get_custom_master_path(csv_name, model=None):
    """
    ローカルmaster CSVの保存先パスを返す。
    """
    model = validate_model(model or FIXED_MODEL)
    csv_name = normalize_csv_name(csv_name, model=model)
    return os.path.join(SAVE_DIR, csv_name)

# =========================================================
# model と work親ディレクトリの整合性チェック
# =========================================================

MODEL_REMOTE_DIR_CONFIG = {
    "gbsw": {
        "subdir_prefix": "gbsw",
        "analysis_prefix": "ana_gbsw",
    },
    "hdgb": {
        "subdir_prefix": "hdgb",
        "analysis_prefix": "ana_hdgb",
    },
    "hdgbvdw": {
        "subdir_prefix": "hdgbvdw",
        "analysis_prefix": "ana_hdgbvdw",
    },
}



def check_remote_model_dirs(client, model, base_dir, start_id, end_id):
    """
    サーバー上で、選択したmodelに対応する
      model_dir: hdgb_1 / hdgbvdw_1 / gbsw_1
      analysis_dir: ana_hdgb_1 / ana_hdgbvdw_1 / ana_gbsw_1
    が実際に存在するか確認する。

    範囲内で最初に見つかったworkを使ってチェックする。
    """
    model = validate_model(model)
    cfg = MODEL_REMOTE_DIR_CONFIG[model]

    subdir_prefix = cfg["subdir_prefix"]
    analysis_prefix = cfg["analysis_prefix"]

    start_id = int(start_id)
    end_id = int(end_id)

    cmd = f"""
base='{base_dir}'
start_id={start_id}
end_id={end_id}
subdir_prefix='{subdir_prefix}'
analysis_prefix='{analysis_prefix}'

echo '===== MODEL/BASE CHECK ====='
echo "model={model}"
echo "base=$base"
echo "subdir_prefix=$subdir_prefix"
echo "analysis_prefix=$analysis_prefix"

if [ ! -d "$base" ]; then
    echo "__BASE_NOT_FOUND__"
    exit 0
fi

found_work=""

for i in $(seq "$start_id" "$end_id"); do
    w=$(find "$base" -maxdepth 1 -type d -name "${{i}}_*work" 2>/dev/null | head -n 1)
    if [ -n "$w" ]; then
        found_work="$w"
        break
    fi
done

if [ -z "$found_work" ]; then
    echo "__NO_WORK_IN_RANGE__"
    exit 0
fi

echo "found_work=$found_work"

if [ -d "$found_work/${{subdir_prefix}}_1" ]; then
    echo "__MD_DIR_OK__"
else
    echo "__MD_DIR_NG__"
    echo "missing: $found_work/${{subdir_prefix}}_1"
fi

if [ -d "$found_work/analysis/${{analysis_prefix}}_1" ]; then
    echo "__ANALYSIS_DIR_OK__"
else
    echo "__ANALYSIS_DIR_NG__"
    echo "missing: $found_work/analysis/${{analysis_prefix}}_1"
fi
"""

    stdin, stdout, stderr = client.exec_command(cmd)
    out = stdout.read().decode()
    err = stderr.read().decode()

    print(out)

    if err:
        print("stderr:")
        print(err)

    if "__BASE_NOT_FOUND__" in out:
        raise ValueError(
            "指定したwork親ディレクトリがサーバー上に存在しません。\n"
            f"base_dir: {base_dir}"
        )

    if "__NO_WORK_IN_RANGE__" in out:
        raise ValueError(
            "指定ID範囲内にworkディレクトリが見つかりません。\n"
            f"base_dir: {base_dir}\n"
            f"ID範囲: {start_id} ~ {end_id}"
        )

    if "__MD_DIR_NG__" in out:
        raise ValueError(
            "選択したmodelに対応する計算ディレクトリが見つかりません。\n"
            f"model: {model}\n"
            f"期待: {subdir_prefix}_1\n"
            "model と work親ディレクトリの組み合わせがズレている可能性が高いです。"
        )

    if "__ANALYSIS_DIR_NG__" in out:
        raise ValueError(
            "選択したmodelに対応するanalysisディレクトリが見つかりません。\n"
            f"model: {model}\n"
            f"期待: analysis/{analysis_prefix}_1\n"
            "model と work親ディレクトリの組み合わせを確認してください。"
        )

    return True

def submit_md_aggregation_background_custom_name(
    client,
    model,
    base_dir,
    start_id,
    end_id,
    target_date="ALL",
    target_t_end_ps="500",
    skip_ps=250.0,
    dt_ps=1.0,
    tolerance_frames=0,
    save_csv_name=None,
):
    """
    サーバー側で集計をバックグラウンド実行。
    ローカル保存用CSV名を指定できる版。

    安全版：
      - model と save_csv_name の食い違いを防ぐ
      - base_dir名には依存しない
      - modelに応じて、work内の subdir / analysis dir だけ確認する
    """

    # 注意：
    # ここでは install_remote_universal_aggregator は呼ばない。
    # 古いスクリプトで上書きしてしまう可能性があるため。

    model = validate_model(model)

    # -----------------------------------------------------
    # 1. サーバー上で実際のwork構造を確認
    # base_dir名ではなく、work内の
    #   gbsw_1 / hdgb_1 / hdgbvdw_1
    #   analysis/ana_gbsw_1 / ana_hdgb_1 / ana_hdgbvdw_1
    # を見る
    # -----------------------------------------------------

    check_remote_model_dirs(
        client=client,
        model=model,
        base_dir=base_dir,
        start_id=start_id,
        end_id=end_id
    )

    # -----------------------------------------------------
    # 2. 保存CSV名の整合性チェック
    # -----------------------------------------------------

    save_csv_name = normalize_csv_name(
        save_csv_name or f"master_database_{model}.csv",
        model=model
    )

    save_csv_name = check_model_csv_consistency(
        model,
        save_csv_name
    )

    base_name = save_csv_name.replace(".csv", "")

    tag_date = str(target_date).replace("*", "ALL").replace("/", "_")
    tag_tend = str(target_t_end_ps).replace("*", "ALL")
    timestamp = time.strftime("%Y%m%d_%H%M%S")

    remote_csv = (
        f"{REMOTE_OUT_DIR}/{base_name}_"
        f"{model}_ID{start_id}_{end_id}_{tag_date}_{tag_tend}ps_{timestamp}.csv"
    )

    remote_log = remote_csv.replace(".csv", ".log")

    cmd = (
        f"nohup python3 {REMOTE_SCRIPT_PATH} "
        f"--model '{model}' "
        f"--base_dir '{base_dir}' "
        f"--start {start_id} "
        f"--end {end_id} "
        f"--date '{target_date}' "
        f"--tend '{target_t_end_ps}' "
        f"--skip {skip_ps} "
        f"--dt {dt_ps} "
        f"--tolerance_frames {tolerance_frames} "
        f"--out '{remote_csv}' "
        f"> '{remote_log}' 2>&1 & echo $!"
    )

    stdin, stdout, stderr = client.exec_command(cmd)
    pid = stdout.read().decode().strip()
    err = stderr.read().decode().strip()

    if err:
        print("stderr:", err)

    print("🚀 集計をバックグラウンド投入しました")
    print("model:", model)
    print("base_dir:", base_dir)
    print("PID:", pid)
    print("remote_csv:", remote_csv)
    print("remote_log:", remote_log)
    print("local_save_csv:", save_csv_name)

    if f"_{model}_ID" not in remote_csv:
        raise RuntimeError(
            "remote_csv名にmodel名が正しく入っていません。\n"
            f"model: {model}\n"
            f"remote_csv: {remote_csv}"
        )

    return {
        "model": model,
        "pid": pid,
        "remote_csv": remote_csv,
        "remote_log": remote_log,
        "base_dir": base_dir,
        "save_csv_name": save_csv_name,
    }

def download_and_merge_md_aggregation_custom_name(client, job):
    """
    完了CSVをダウンロードし、指定したCSV名に統合する。
    """

    model = validate_model(job["model"])
    remote_csv = job["remote_csv"]

    save_csv_name = normalize_csv_name(
        job.get("save_csv_name", f"master_database_{model}.csv"),
        model=model
    )

    save_csv_name = check_model_csv_consistency(model, save_csv_name)

    local_master_csv = get_custom_master_path(save_csv_name, model=model)

    local_tmp = os.path.join(
        SAVE_DIR,
        os.path.basename(remote_csv)
    )

    sftp = client.open_sftp()
    try:
        sftp.get(remote_csv, local_tmp)
    finally:
        sftp.close()

    print("✅ downloaded temporary csv:")
    print(local_tmp)

    new_df = pd.read_csv(local_tmp)

    # 念のため、CSV内のModel列も確認
    if "Model" in new_df.columns:
        actual_models = set(
            new_df["Model"].dropna().astype(str).str.lower().unique()
        )

        if actual_models and actual_models != {model}:
            raise ValueError(
                "ダウンロードしたCSV内のModel列が想定と違います。\n"
                f"expected model: {model}\n"
                f"actual models : {actual_models}\n"
                "このCSVは統合しない方が安全です。"
            )

    if os.path.exists(local_master_csv):
        old_df = pd.read_csv(local_master_csv)

        final_df = pd.concat([old_df, new_df], ignore_index=True)

        final_df = final_df.drop_duplicates(
            subset=["Model", "ID", "Date", "Pos_Num", "t_end_ps"],
            keep="last"
        )
    else:
        final_df = new_df

    # 数値列を数値化してからsort
    for col in ["ID", "Pos_Num", "t_end_ps"]:
        if col in final_df.columns:
            final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

    final_df = final_df.sort_values(
        ["ID", "Date", "t_end_ps", "Pos_Num"]
    ).reset_index(drop=True)

    final_df.to_csv(local_master_csv, index=False)

    print("✅ merged master saved:")
    print(local_master_csv)
    print("master shape:", final_df.shape)

    if "Model" in final_df.columns:
        print("\nModel counts:")
        print(final_df["Model"].value_counts(dropna=False))

    if "Status" in final_df.columns:
        print("\nStatus counts:")
        print(final_df["Status"].value_counts(dropna=False))

    display(final_df.tail(10))

    return final_df


def check_md_aggregation_job(client, job):
    """
    バックグラウンド集計ジョブの進捗確認。
    """
    pid = job.get("pid")
    remote_csv = job["remote_csv"]
    remote_log = job["remote_log"]

    if pid is None or str(pid).strip() == "":
        pid_check = "echo 'PIDなし'"
    else:
        pid_check = f"ps -p {pid} -o pid,stat,etime,cmd"

    cmd = f"""
echo '===== PID CHECK ====='
{pid_check}

echo ''
echo '===== CSV CHECK ====='
ls -lh '{remote_csv}' 2>/dev/null || echo 'CSV not created yet'

echo ''
echo '===== LOG CHECK ====='
ls -lh '{remote_log}' 2>/dev/null || echo 'LOG not created yet'

echo ''
echo '===== LOG TAIL ====='
tail -n 80 '{remote_log}' 2>/dev/null || echo 'No log yet'

echo ''
echo '===== CSV HEAD ====='
head -n 3 '{remote_csv}' 2>/dev/null || echo 'No csv yet'

echo ''
echo '===== CSV LINE COUNT ====='
wc -l '{remote_csv}' 2>/dev/null || echo 'No csv yet'
"""

    stdin, stdout, stderr = client.exec_command(cmd)

    out = stdout.read().decode()
    err = stderr.read().decode()

    print(out)

    if err:
        print("stderr:")
        print(err)


print("✅ safe MD aggregation helper loaded")
print("FIXED_MODEL:", FIXED_MODEL)
print("default save csv:", normalize_csv_name("", model=FIXED_MODEL))


✅ safe MD aggregation helper loaded
FIXED_MODEL: hdgb
default save csv: master_database_hdgb.csv


In [7]:
# =========================================================
# modelに応じて、work内で探すディレクトリ名を決める
# base_dirの名前には依存しない版
# =========================================================

MODEL_REMOTE_DIR_CONFIG = {
    "gbsw": {
        "subdir_prefix": "gbsw",
        "analysis_prefix": "ana_gbsw",
    },
    "hdgb": {
        "subdir_prefix": "hdgb",
        "analysis_prefix": "ana_hdgb",
    },
    "hdgbvdw": {
        "subdir_prefix": "hdgbvdw",
        "analysis_prefix": "ana_hdgbvdw",
    },
}


def check_remote_model_dirs(client, model, base_dir, start_id, end_id):
    """
    base_dirの名前ではなく、
    選択したmodelに対応するwork内ディレクトリがあるかだけ確認する。

    gbsw:
      gbsw_1
      analysis/ana_gbsw_1

    hdgb:
      hdgb_1
      analysis/ana_hdgb_1

    hdgbvdw:
      hdgbvdw_1
      analysis/ana_hdgbvdw_1

    ※ hdgbvdw work内に analysis/ana_hdgb_1 があっても無視する。
    """

    model = validate_model(model)
    cfg = MODEL_REMOTE_DIR_CONFIG[model]

    subdir_prefix = cfg["subdir_prefix"]
    analysis_prefix = cfg["analysis_prefix"]

    start_id = int(start_id)
    end_id = int(end_id)

    cmd = f"""
base='{base_dir}'
start_id={start_id}
end_id={end_id}
subdir_prefix='{subdir_prefix}'
analysis_prefix='{analysis_prefix}'

echo '===== MODEL/WORK STRUCTURE CHECK ====='
echo "model=$'{model}'"
echo "base=$base"
echo "expected md dir: ${{subdir_prefix}}_1"
echo "expected analysis dir: analysis/${{analysis_prefix}}_1"

if [ ! -d "$base" ]; then
    echo "__BASE_NOT_FOUND__"
    exit 0
fi

found_work=""

for i in $(seq "$start_id" "$end_id"); do
    w=$(find "$base" -maxdepth 1 -type d -name "${{i}}_*work" 2>/dev/null | head -n 1)
    if [ -n "$w" ]; then
        found_work="$w"
        break
    fi
done

if [ -z "$found_work" ]; then
    echo "__NO_WORK_IN_RANGE__"
    exit 0
fi

echo "found_work=$found_work"

if [ -d "$found_work/${{subdir_prefix}}_1" ]; then
    echo "__MD_DIR_OK__"
else
    echo "__MD_DIR_NG__"
    echo "missing: $found_work/${{subdir_prefix}}_1"
fi

if [ -d "$found_work/analysis/${{analysis_prefix}}_1" ]; then
    echo "__ANALYSIS_DIR_OK__"
else
    echo "__ANALYSIS_DIR_NG__"
    echo "missing: $found_work/analysis/${{analysis_prefix}}_1"
fi
"""

    stdin, stdout, stderr = client.exec_command(cmd)
    out = stdout.read().decode()
    err = stderr.read().decode()

    print(out)

    if err:
        print("stderr:")
        print(err)

    if "__BASE_NOT_FOUND__" in out:
        raise ValueError(
            "指定したwork親ディレクトリがサーバー上に存在しません。\n"
            f"base_dir: {base_dir}"
        )

    if "__NO_WORK_IN_RANGE__" in out:
        raise ValueError(
            "指定ID範囲内にworkディレクトリが見つかりません。\n"
            f"base_dir: {base_dir}\n"
            f"ID範囲: {start_id} ~ {end_id}"
        )

    if "__MD_DIR_NG__" in out:
        raise ValueError(
            "選択したmodelに対応する計算ディレクトリが見つかりません。\n"
            f"model: {model}\n"
            f"期待: {subdir_prefix}_1\n"
            "base_dir内のworkが、選択したmodelの計算結果ではない可能性があります。"
        )

    if "__ANALYSIS_DIR_NG__" in out:
        raise ValueError(
            "選択したmodelに対応するanalysisディレクトリが見つかりません。\n"
            f"model: {model}\n"
            f"期待: analysis/{analysis_prefix}_1\n"
            "解析ディレクトリ名を確認してください。"
        )

    return True


def submit_md_aggregation_background_custom_name(
    client,
    model,
    base_dir,
    start_id,
    end_id,
    target_date="ALL",
    target_t_end_ps="500",
    skip_ps=250.0,
    dt_ps=1.0,
    tolerance_frames=0,
    save_csv_name=None,
):
    """
    サーバー側で集計をバックグラウンド実行。
    ローカル保存用CSV名を指定できる版。

    安全版：
      - model と save_csv_name の食い違いを防ぐ
      - base_dir名には依存しない
      - modelに応じて、work内の subdir / analysis dir だけ確認する
    """

    # 注意：
    # ここでは install_remote_universal_aggregator は呼ばない。
    # 古いスクリプトで上書きしてしまう可能性があるため。

    model = validate_model(model)

    # -----------------------------------------------------
    # 1. サーバー上で実際のwork構造を確認
    # base_dir名ではなく、work内の
    #   gbsw_1 / hdgb_1 / hdgbvdw_1
    #   analysis/ana_gbsw_1 / ana_hdgb_1 / ana_hdgbvdw_1
    # を見る
    # -----------------------------------------------------

    check_remote_model_dirs(
        client=client,
        model=model,
        base_dir=base_dir,
        start_id=start_id,
        end_id=end_id
    )

    # -----------------------------------------------------
    # 2. 保存CSV名の整合性チェック
    # -----------------------------------------------------

    save_csv_name = normalize_csv_name(
        save_csv_name or f"master_database_{model}.csv",
        model=model
    )

    save_csv_name = check_model_csv_consistency(
        model,
        save_csv_name
    )

    base_name = save_csv_name.replace(".csv", "")

    tag_date = str(target_date).replace("*", "ALL").replace("/", "_")
    tag_tend = str(target_t_end_ps).replace("*", "ALL")
    timestamp = time.strftime("%Y%m%d_%H%M%S")

    remote_csv = (
        f"{REMOTE_OUT_DIR}/{base_name}_"
        f"{model}_ID{start_id}_{end_id}_{tag_date}_{tag_tend}ps_{timestamp}.csv"
    )

    remote_log = remote_csv.replace(".csv", ".log")

    cmd = (
        f"nohup python3 {REMOTE_SCRIPT_PATH} "
        f"--model '{model}' "
        f"--base_dir '{base_dir}' "
        f"--start {start_id} "
        f"--end {end_id} "
        f"--date '{target_date}' "
        f"--tend '{target_t_end_ps}' "
        f"--skip {skip_ps} "
        f"--dt {dt_ps} "
        f"--tolerance_frames {tolerance_frames} "
        f"--out '{remote_csv}' "
        f"> '{remote_log}' 2>&1 & echo $!"
    )

    stdin, stdout, stderr = client.exec_command(cmd)
    pid = stdout.read().decode().strip()
    err = stderr.read().decode().strip()

    if err:
        print("stderr:", err)

    print("🚀 集計をバックグラウンド投入しました")
    print("model:", model)
    print("base_dir:", base_dir)
    print("PID:", pid)
    print("remote_csv:", remote_csv)
    print("remote_log:", remote_log)
    print("local_save_csv:", save_csv_name)

    if f"_{model}_ID" not in remote_csv:
        raise RuntimeError(
            "remote_csv名にmodel名が正しく入っていません。\n"
            f"model: {model}\n"
            f"remote_csv: {remote_csv}"
        )

    return {
        "model": model,
        "pid": pid,
        "remote_csv": remote_csv,
        "remote_log": remote_log,
        "base_dir": base_dir,
        "save_csv_name": save_csv_name,
    }


In [8]:
# =========================================================
# モデル選択版 UI
# 安全版 helper 対応
# GBSW / HDGB / HDGBvdW をUIで選択できる版
# =========================================================


current_job = {"job": None}

style = {"description_width": "initial"}


# =========================================================
# モデルごとのデフォルトwork親ディレクトリ
# 必要ならUI上で変更してOK
# =========================================================

DEFAULT_BASE_DIR_BY_MODEL = {
    "gbsw": "/home/teraimao/experiment/confirm",
    "hdgb": "/home/teraimao/experiment/confirm_hdgb_500ps_z軸0,7,15,17,20",
    "hdgbvdw": "/home/teraimao/experiment/confirm_hdgbvdw_500ps_一定_z軸0,7,15,17,20",
}


def default_base_dir_for_model(model):
    model = validate_model(model)
    return DEFAULT_BASE_DIR_BY_MODEL.get(
        model,
        "/home/teraimao/experiment/confirm"
    )


def default_save_csv_for_model(model):
    model = validate_model(model)
    return normalize_csv_name(
        f"master_database_{model}.csv",
        model=model
    )


# =========================================================
# UI部品
# =========================================================

model_box = widgets.Dropdown(
    options=[
        ("GBSW", "gbsw"),
        ("HDGB", "hdgb"),
        ("HDGBvdW", "hdgbvdw"),
    ],
    value=validate_model(globals().get("FIXED_MODEL", "hdgb")),
    description="モデル:",
    style=style,
    layout=widgets.Layout(width="260px")
)

txt_base_dir = widgets.Text(
    value=default_base_dir_for_model(model_box.value),
    description="work親ディレクトリ:",
    style=style,
    layout=widgets.Layout(width="760px")
)

txt_save_csv = widgets.Text(
    value=default_save_csv_for_model(model_box.value),
    description="保存CSV名:",
    style=style,
    layout=widgets.Layout(width="520px")
)

txt_start = widgets.IntText(
    value=1,
    description="開始ID:",
    style=style,
    layout=widgets.Layout(width="160px")
)

txt_end = widgets.IntText(
    value=100,
    description="終了ID:",
    style=style,
    layout=widgets.Layout(width="160px")
)

txt_date = widgets.Text(
    value="ALL",
    description="日付 (ALLで全日程):",
    style=style,
    layout=widgets.Layout(width="260px")
)

txt_tend = widgets.Text(
    value="500",
    description="計算時間ps (ALLで全件):",
    style=style,
    layout=widgets.Layout(width="260px")
)

txt_skip = widgets.FloatText(
    value=250.0,
    description="skip_ps:",
    style=style,
    layout=widgets.Layout(width="180px")
)

txt_dt = widgets.FloatText(
    value=1.0,
    description="dt_ps:",
    style=style,
    layout=widgets.Layout(width="160px")
)

txt_tol = widgets.IntText(
    value=0,
    description="許容欠損frame:",
    style=style,
    layout=widgets.Layout(width="200px")
)

btn_submit = widgets.Button(
    description="🚀 集計をバックグラウンド開始",
    button_style="danger",
    layout=widgets.Layout(width="300px", height="42px")
)

btn_check = widgets.Button(
    description="📡 進捗確認",
    button_style="info",
    layout=widgets.Layout(width="160px", height="42px")
)

btn_download = widgets.Button(
    description="📥 完了CSVを取得して指定CSVに統合",
    button_style="success",
    layout=widgets.Layout(width="340px", height="42px")
)

btn_show_master = widgets.Button(
    description="📄 保存CSV確認",
    button_style="",
    layout=widgets.Layout(width="180px", height="42px")
)

out_log = widgets.Output()


# =========================================================
# モデル変更時にwork親dirと保存CSV名を自動更新
# =========================================================

def on_model_changed(change):
    if change["name"] != "value":
        return

    model = validate_model(change["new"])

    # work親ディレクトリは自動変更しない
    # ディレクトリ名は毎回変わるので、ユーザーが手入力した値を残す
    txt_save_csv.value = default_save_csv_for_model(model)

    current_job["job"] = None

    with out_log:
        clear_output()
        print("✅ モデルを変更しました")
        print("model:", model)
        print("work親ディレクトリは自動変更していません。必要なら手入力してください。")
        print("現在のwork親ディレクトリ:", txt_base_dir.value)
        print("保存CSV名:", txt_save_csv.value)
        print("※ 前のjob情報はリセットしました。")


# =========================================================
# ボタン動作
# =========================================================

def on_submit_clicked(b):
    with out_log:
        clear_output()

        try:
            model = validate_model(model_box.value)

            # グローバルにも反映しておく
            globals()["FIXED_MODEL"] = model

            save_csv_name = normalize_csv_name(
                txt_save_csv.value,
                model=model
            )

            save_csv_name = check_model_csv_consistency(
                model,
                save_csv_name
            )

            local_save_path = get_custom_master_path(
                save_csv_name,
                model=model
            )

            print("🚀 MD解析データ集計をバックグラウンドで開始します")
            print(f"モデル        : {model}")
            print(f"work親dir     : {txt_base_dir.value}")
            print(f"保存CSV名     : {save_csv_name}")
            print(f"ローカル保存先 : {local_save_path}")
            print(f"ID範囲        : {txt_start.value} ~ {txt_end.value}")
            print(f"日付          : {txt_date.value}")
            print(f"計算時間ps    : {txt_tend.value}")
            print(f"skip_ps       : {txt_skip.value}")
            print(f"dt_ps         : {txt_dt.value}")
            print(f"許容欠損      : {txt_tol.value}")
            print("-" * 90)

            job = submit_md_aggregation_background_custom_name(
                client=client,
                model=model,
                base_dir=txt_base_dir.value,
                start_id=txt_start.value,
                end_id=txt_end.value,
                target_date=txt_date.value,
                target_t_end_ps=txt_tend.value,
                skip_ps=txt_skip.value,
                dt_ps=txt_dt.value,
                tolerance_frames=txt_tol.value,
                save_csv_name=save_csv_name,
            )

            current_job["job"] = job

            print("\n✅ 投入完了。Notebookはこのまま他のセルを実行できます。")
            print("\n===== JOB INFO =====")
            print("model      :", job["model"])
            print("PID        :", job["pid"])
            print("remote_csv :", job["remote_csv"])
            print("remote_log :", job["remote_log"])
            print("save_csv   :", job["save_csv_name"])

            if f"_{model}_ID" in job["remote_csv"]:
                print("\n✅ remote_csv のmodel名チェックOK")
            else:
                print("\n⚠️ remote_csv のmodel名が怪しいです。確認してください。")

        except Exception as e:
            print("❌ エラー:", e)


def on_check_clicked(b):
    with out_log:
        clear_output()

        job = current_job.get("job")

        if job is None:
            print("⚠️ まだjobが投入されていません。")
            return

        print("📡 進捗確認中...")
        print("-" * 90)

        try:
            check_md_aggregation_job(client, job)
        except Exception as e:
            print("❌ エラー:", e)


def on_download_clicked(b):
    with out_log:
        clear_output()

        job = current_job.get("job")

        if job is None:
            print("⚠️ まだjobが投入されていません。")
            return

        print("📥 CSVをダウンロードして指定CSVに統合します")
        print("-" * 90)

        try:
            df_master = download_and_merge_md_aggregation_custom_name(
                client=client,
                job=job,
            )

            print("\n✅ 統合完了")
            print("shape:", df_master.shape)

        except Exception as e:
            print("❌ エラー:", e)


def on_show_master_clicked(b):
    with out_log:
        clear_output()

        try:
            model = validate_model(model_box.value)

            save_csv_name = normalize_csv_name(
                txt_save_csv.value,
                model=model
            )

            save_csv_name = check_model_csv_consistency(
                model,
                save_csv_name
            )

            master_path = get_custom_master_path(
                save_csv_name,
                model=model
            )

            if not os.path.exists(master_path):
                print("⚠️ 指定CSVがまだありません。")
                print(master_path)
                return

            df = pd.read_csv(master_path)

            print("📄 保存CSV確認")
            print("path :", master_path)
            print("shape:", df.shape)

            if "Model" in df.columns:
                print("\nModel counts:")
                print(df["Model"].value_counts(dropna=False))

            if "Status" in df.columns:
                print("\nStatus counts:")
                print(df["Status"].value_counts(dropna=False))

            if "ID" in df.columns:
                print("\nID range:")
                print(df["ID"].min(), "~", df["ID"].max())

            if "Date" in df.columns:
                print("\nDate counts:")
                print(df["Date"].value_counts().head())

            if "t_end_ps" in df.columns:
                print("\nt_end_ps counts:")
                print(df["t_end_ps"].value_counts().head())

            if "LogPath" in df.columns:
                print("\nLogPath check:")
                print("hdgbvdw in LogPath:", df["LogPath"].astype(str).str.contains("hdgbvdw").sum())
                print("hdgb_ in LogPath:", df["LogPath"].astype(str).str.contains("/hdgb_").sum())
                print("gbsw_ in LogPath:", df["LogPath"].astype(str).str.contains("/gbsw_").sum())

            display(df.tail(10))

        except Exception as e:
            print("❌ エラー:", e)


# =========================================================
# ボタン接続
# =========================================================

btn_submit.on_click(on_submit_clicked)
btn_check.on_click(on_check_clicked)
btn_download.on_click(on_download_clicked)
btn_show_master.on_click(on_show_master_clicked)


# =========================================================
# UI表示
# =========================================================

ui = widgets.VBox([
    widgets.HTML("<h3>🐚 MD解析データ集計：モデル選択・保存CSV名指定版</h3>"),
    widgets.HTML(
        "<p>モデルをプルダウンで選択できます。"
        "保存CSV名とwork親ディレクトリはモデル変更時に自動更新されます。</p>"
    ),
    widgets.HBox([model_box]),
    widgets.HBox([txt_base_dir]),
    widgets.HBox([txt_save_csv]),
    widgets.HBox([txt_start, txt_end, txt_date, txt_tend]),
    widgets.HBox([txt_skip, txt_dt, txt_tol]),
    widgets.HBox([btn_submit, btn_check]),
    widgets.HBox([btn_download, btn_show_master]),
    out_log
])

display(ui)

print("✅ model-select aggregation UI loaded")
print("selected model:", model_box.value)
print("default work親dir:", txt_base_dir.value)
print("default save csv:", txt_save_csv.value)


✅ model-select aggregation UI loaded
selected model: hdgb
default work親dir: /home/teraimao/experiment/confirm_hdgb_500ps_z軸0,7,15,17,20
default save csv: master_database_hdgb.csv


### 終わり

## 更新して current_job が消えたときの復旧セル

In [9]:
# =========================================================
# 更新して current_job が消えたときの復旧セル
# サーバー上の aggregation_results から最新CSV/logを探して統合する
# =========================================================


SAVE_DIR = "/Users/teraimao/experiment/data/解析結果/"
os.makedirs(SAVE_DIR, exist_ok=True)

REMOTE_OUT_DIR = "/home/teraimao/experiment/aggregation_results"

# ここを使いたいモデルにする
# "gbsw", "hdgb", "hdgbvdw"
FIXED_MODEL = "hdgbvdw"

# 統合先CSV
SAVE_CSV_NAME = f"master_database_{FIXED_MODEL}.csv"


def normalize_csv_name(name):
    name = str(name).strip()
    if name == "":
        name = f"master_database_{FIXED_MODEL}.csv"
    name = os.path.basename(name)
    if not name.endswith(".csv"):
        name += ".csv"
    return name


def get_custom_master_path(csv_name):
    csv_name = normalize_csv_name(csv_name)
    return os.path.join(SAVE_DIR, csv_name)


def ensure_client():
    """
    client が残っていればそれを使う。
    なければ connect_server() で再接続。
    """
    global client

    if "client" in globals() and client is not None:
        try:
            client.exec_command("echo ok")
            return client
        except Exception:
            pass

    if "connect_server" not in globals():
        raise RuntimeError("client がなく、connect_server も未定義です。先にSSH接続セルを実行してください。")

    client = connect_server()
    if client is None:
        raise RuntimeError("SSH接続に失敗しました。")

    return client


def list_remote_aggregation_files(client, model=FIXED_MODEL, n=20):
    """
    サーバー上の集計CSV/logを新しい順に表示。
    """
    model = model.lower()

    cmd = f"""
echo '===== latest csv files ====='
ls -lt {REMOTE_OUT_DIR}/*{model}*.csv 2>/dev/null | head -n {n} || echo 'no csv'

echo ''
echo '===== latest log files ====='
ls -lt {REMOTE_OUT_DIR}/*{model}*.log 2>/dev/null | head -n {n} || echo 'no log'

echo ''
echo '===== running aggregator processes ====='
pgrep -af aggregate_md_fast_universal.py || echo 'no running aggregator'
"""

    stdin, stdout, stderr = client.exec_command(cmd)
    out = stdout.read().decode()
    err = stderr.read().decode()

    print(out)
    if err:
        print("stderr:")
        print(err)


def recover_latest_aggregation_job(client, model=FIXED_MODEL, save_csv_name=SAVE_CSV_NAME):
    """
    最新のCSVとlogを探して job dict を復元。
    """
    model = model.lower()
    save_csv_name = normalize_csv_name(save_csv_name)

    cmd = f"ls -t {REMOTE_OUT_DIR}/*{model}*.csv 2>/dev/null | head -n 1"
    stdin, stdout, stderr = client.exec_command(cmd)

    remote_csv = stdout.read().decode().strip()
    err = stderr.read().decode().strip()

    if err:
        print("stderr:", err)

    if not remote_csv:
        raise RuntimeError(f"{REMOTE_OUT_DIR} に {model} のCSVが見つかりません。まだ集計中か、model名が違う可能性があります。")

    remote_log = remote_csv.replace(".csv", ".log")

    job = {
        "model": model,
        "pid": None,
        "remote_csv": remote_csv,
        "remote_log": remote_log,
        "save_csv_name": save_csv_name,
    }

    print("✅ recovered job")
    print("model      :", job["model"])
    print("remote_csv :", job["remote_csv"])
    print("remote_log :", job["remote_log"])
    print("save_csv   :", job["save_csv_name"])

    return job


def check_recovered_aggregation_job(client, job):
    """
    PIDなしでもCSV/logと実行中プロセスを確認。
    """
    remote_csv = job["remote_csv"]
    remote_log = job["remote_log"]
    model = job["model"]

    cmd = f"""
echo '===== running aggregator processes ====='
pgrep -af aggregate_md_fast_universal.py || echo 'no running aggregator'

echo ''
echo '===== CSV CHECK ====='
ls -lh '{remote_csv}' 2>/dev/null || echo 'CSV not created yet'

echo ''
echo '===== LOG CHECK ====='
ls -lh '{remote_log}' 2>/dev/null || echo 'LOG not created yet'

echo ''
echo '===== LOG TAIL ====='
tail -n 80 '{remote_log}' 2>/dev/null || echo 'No log yet'

echo ''
echo '===== CSV HEAD ====='
head -n 3 '{remote_csv}' 2>/dev/null || echo 'No csv yet'

echo ''
echo '===== CSV LINE COUNT ====='
wc -l '{remote_csv}' 2>/dev/null || echo 'No csv yet'
"""

    stdin, stdout, stderr = client.exec_command(cmd)
    out = stdout.read().decode()
    err = stderr.read().decode()

    print(out)
    if err:
        print("stderr:")
        print(err)


def download_and_merge_recovered_aggregation(client, job):
    """
    復元したremote_csvをダウンロードして、指定CSVに統合。
    """
    model = job["model"].lower()
    remote_csv = job["remote_csv"]
    save_csv_name = normalize_csv_name(job.get("save_csv_name", f"master_database_{model}.csv"))

    local_master_csv = get_custom_master_path(save_csv_name)

    local_tmp = os.path.join(
        SAVE_DIR,
        os.path.basename(remote_csv)
    )

    sftp = client.open_sftp()
    try:
        sftp.get(remote_csv, local_tmp)
    finally:
        sftp.close()

    print("✅ downloaded temporary csv:")
    print(local_tmp)

    new_df = pd.read_csv(local_tmp)

    if os.path.exists(local_master_csv):
        old_df = pd.read_csv(local_master_csv)

        final_df = pd.concat([old_df, new_df], ignore_index=True)

        final_df = final_df.drop_duplicates(
            subset=["Model", "ID", "Date", "Pos_Num", "t_end_ps"],
            keep="last"
        )
    else:
        final_df = new_df

    # 数値列を数値化してからsort
    for col in ["ID", "Pos_Num", "t_end_ps"]:
        if col in final_df.columns:
            final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

    final_df = final_df.sort_values(
        ["ID", "Date", "t_end_ps", "Pos_Num"]
    ).reset_index(drop=True)

    final_df.to_csv(local_master_csv, index=False)

    print("✅ merged master saved:")
    print(local_master_csv)
    print("master shape:", final_df.shape)

    print("\nID range:")
    print(final_df["ID"].min(), "~", final_df["ID"].max())

    print("\nID counts tail:")
    print(final_df["ID"].value_counts().sort_index().tail(20))

    display(final_df.tail(10))

    return final_df


# =========================================================
# 実行
# =========================================================

client = ensure_client()

print("まずサーバー上の候補を表示します。")
list_remote_aggregation_files(client, FIXED_MODEL, n=20)

print("\n最新のjobを復元します。")
recovered_job = recover_latest_aggregation_job(
    client,
    model=FIXED_MODEL,
    save_csv_name=SAVE_CSV_NAME
)

print("\n進捗を確認します。")
check_recovered_aggregation_job(client, recovered_job)

まずサーバー上の候補を表示します。
===== latest csv files =====
-rw-rw-r-- 1 teraimao teraimao    3347 Jul  6 15:52 /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgbvdw_ID1_2_ALL_500ps_20260706_155242.csv
-rw-rw-r-- 1 teraimao teraimao 1769347 Jul  6 15:40 /home/teraimao/experiment/aggregation_results/master_database_hdgbvdw_hdgbvdw_ID101_1300_ALL_500ps_20260706_152639.csv
-rw-rw-r-- 1 teraimao teraimao  203353 Jul  6 15:20 /home/teraimao/experiment/aggregation_results/master_database_hdgbvdw_hdgbvdw_ID5_100_ALL_500ps_20260706_151947.csv
-rw-rw-r-- 1 teraimao teraimao    4598 Jul  6 15:18 /home/teraimao/experiment/aggregation_results/master_database_hdgbvdw_hdgbvdw_ID3_4_ALL_500ps_20260706_151843.csv
-rw-rw-r-- 1 teraimao teraimao    9017 Jul  6 15:16 /home/teraimao/experiment/aggregation_results/master_database_hdgbvdw_hdgbvdw_ID0_4_ALL_500ps_20260706_151644.csv
-rw-rw-r-- 1 teraimao teraimao     191 Jul  6 15:11 /home/teraimao/experiment/aggregation_results/master_database_hdgbv

In [10]:
# =========================================================
# HDGB ID7~100 のjobを手動復元して進捗確認
# =========================================================

job_hdgb_7_100 = {
    "model": "hdgb",
    "pid": "2676716",
    "remote_csv": "/home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID7_100_ALL_500ps_20260706_183448.csv",
    "remote_log": "/home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID7_100_ALL_500ps_20260706_183448.log",
    "base_dir": "/home/teraimao/experiment/confirm_hdgb_500ps_z軸0,7,15,17,20",
    "save_csv_name": "master_database_hdgb.csv",
}

current_job["job"] = job_hdgb_7_100

check_md_aggregation_job(client, job_hdgb_7_100)

===== PID CHECK =====
    PID STAT     ELAPSED CMD

===== CSV CHECK =====
-rw-rw-r-- 1 teraimao teraimao 240K Jul  6 18:36 /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID7_100_ALL_500ps_20260706_183448.csv

===== LOG CHECK =====
-rw-rw-r-- 1 teraimao teraimao 2.0K Jul  6 18:36 /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID7_100_ALL_500ps_20260706_183448.log

===== LOG TAIL =====
[27] done, works=1
[28] done, works=1
[29] done, works=1
[30] done, works=1
[31] done, works=1
[32] done, works=1
[33] done, works=1
[34] done, works=1
[35] done, works=1
[36] done, works=1
[37] done, works=1
[38] done, works=1
[39] done, works=1
[40] done, works=1
[41] done, works=1
[42] no work dirs
[43] done, works=1
[44] no work dirs
[45] no work dirs
[46] no work dirs
[47] no work dirs
[48] done, works=1
[49] done, works=1
[50] done, works=1
[51] done, works=1
[52] done, works=1
[53] done, works=1
[54] done, works=1
[55] done, works=1
[56] done, works=

In [14]:
# =========================================================
# HDGB 101~1000 の最新jobをサーバー上から探して復元・進捗確認
# =========================================================

import os
import pandas as pd

remote_pattern_csv = "/home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_*.csv"
remote_pattern_log = "/home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_*.log"

cmd = f"""
echo '===== HDGB 101~1000 CSV candidates ====='
ls -lt {remote_pattern_csv} 2>/dev/null | head -n 10 || echo 'no csv'

echo ''
echo '===== HDGB 101~1000 LOG candidates ====='
ls -lt {remote_pattern_log} 2>/dev/null | head -n 10 || echo 'no log'

echo ''
echo '===== running aggregator processes ====='
pgrep -af aggregate_md_fast_universal.py || echo 'no running aggregator'
"""

stdin, stdout, stderr = client.exec_command(cmd)
out = stdout.read().decode()
err = stderr.read().decode()

print(out)

if err:
    print("stderr:")
    print(err)

===== HDGB 101~1000 CSV candidates =====
-rw-rw-r-- 1 teraimao teraimao 191 Jul  6 18:37 /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_183731.csv

===== HDGB 101~1000 LOG candidates =====
-rw-rw-r-- 1 teraimao teraimao 17263 Jul  6 18:37 /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_183731.log

===== running aggregator processes =====
2677876 bash -c  echo '===== HDGB 101~1000 CSV candidates =====' ls -lt /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_*.csv 2>/dev/null | head -n 10 || echo 'no csv'  echo '' echo '===== HDGB 101~1000 LOG candidates =====' ls -lt /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_*.log 2>/dev/null | head -n 10 || echo 'no log'  echo '' echo '===== running aggregator processes =====' pgrep -af aggregate_md_fast_universal.py || echo 'no running aggregator' 



In [15]:
# =========================================================
# HDGB 101~1000 のCSV/logの中身を確認
# =========================================================

job_hdgb_101_1000 = {
    "model": "hdgb",
    "pid": None,
    "remote_csv": "/home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_183731.csv",
    "remote_log": "/home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_183731.log",
    "base_dir": "/home/teraimao/experiment/confirm_hdgb101",
    "save_csv_name": "master_database_hdgb.csv",
}

current_job["job"] = job_hdgb_101_1000

check_md_aggregation_job(client, job_hdgb_101_1000)

===== PID CHECK =====
PIDなし

===== CSV CHECK =====
-rw-rw-r-- 1 teraimao teraimao 191 Jul  6 18:37 /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_183731.csv

===== LOG CHECK =====
-rw-rw-r-- 1 teraimao teraimao 17K Jul  6 18:37 /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_183731.log

===== LOG TAIL =====
[925] no work dirs
[926] no work dirs
[927] no work dirs
[928] no work dirs
[929] no work dirs
[930] no work dirs
[931] no work dirs
[932] no work dirs
[933] no work dirs
[934] no work dirs
[935] no work dirs
[936] no work dirs
[937] no work dirs
[938] no work dirs
[939] no work dirs
[940] no work dirs
[941] no work dirs
[942] no work dirs
[943] no work dirs
[944] no work dirs
[945] no work dirs
[946] no work dirs
[947] no work dirs
[948] no work dirs
[949] no work dirs
[950] no work dirs
[951] no work dirs
[952] no work dirs
[953] no work dirs
[954] no work dirs
[955] no wor

In [16]:
# =========================================================
# confirm_hdgb101 を集計スクリプトが読める形にする
# 元データはコピーせず、集計用リンクディレクトリを作る
# =========================================================

SRC_BASE = "/home/teraimao/experiment/confirm_hdgb101"
LINK_BASE = "/home/teraimao/experiment/confirm_hdgb101_for_aggregation"

cmd = f"""
set -e

SRC_BASE="{SRC_BASE}"
LINK_BASE="{LINK_BASE}"

mkdir -p "$LINK_BASE"

echo "source base: $SRC_BASE"
echo "link base  : $LINK_BASE"
echo ""

linked=0
missing=0

for i in $(seq 101 1000); do
    orig=$(find "$SRC_BASE" -maxdepth 1 -type d \\( -name "${{i}}_*_500ps_hdgb_work" -o -name "${{i}}_*_500ps_work" \\) 2>/dev/null | head -n 1)

    if [ -z "$orig" ]; then
        echo "[$i] no source work"
        missing=$((missing + 1))
        continue
    fi

    base=$(basename "$orig")

    # 101_2026.06.30_500ps_hdgb_work
    # → 101_2026.06.30_500ps_work
    newbase="${{base/_500ps_hdgb_work/_500ps_work}}"

    link="$LINK_BASE/$newbase"

    mkdir -p "$link"

    for d in hdgb_1 hdgb_2 hdgb_3 hdgb_4 hdgb_5 analysis; do
        if [ -e "$orig/$d" ]; then
            ln -sfn "$orig/$d" "$link/$d"
        fi
    done

    linked=$((linked + 1))
done

echo ""
echo "linked works : $linked"
echo "missing works: $missing"

echo ""
echo "===== example link dirs ====="
find "$LINK_BASE" -maxdepth 1 -type d -name "101_*work" -print

echo ""
echo "===== inside example ====="
w=$(find "$LINK_BASE" -maxdepth 1 -type d -name "101_*work" | head -n 1)
echo "work=$w"

if [ -n "$w" ]; then
    ls -l "$w" | head -n 30
    echo ""
    echo "hdgb_1 exists?"
    test -d "$w/hdgb_1" && echo "OK: hdgb_1" || echo "NG: hdgb_1"
    echo "analysis/ana_hdgb_1 exists?"
    test -d "$w/analysis/ana_hdgb_1" && echo "OK: ana_hdgb_1" || echo "NG: ana_hdgb_1"
fi
"""

stdin, stdout, stderr = client.exec_command(cmd)
print(stdout.read().decode())

err = stderr.read().decode()
if err:
    print("stderr:")
    print(err)

source base: /home/teraimao/experiment/confirm_hdgb101
link base  : /home/teraimao/experiment/confirm_hdgb101_for_aggregation

[550] no source work
[554] no source work
[556] no source work
[557] no source work
[558] no source work
[559] no source work
[585] no source work
[586] no source work
[592] no source work
[593] no source work
[594] no source work
[595] no source work
[596] no source work
[598] no source work
[599] no source work
[600] no source work
[601] no source work
[602] no source work
[603] no source work
[604] no source work
[605] no source work
[613] no source work
[614] no source work
[615] no source work
[616] no source work
[622] no source work
[623] no source work
[624] no source work
[625] no source work
[739] no source work
[740] no source work
[787] no source work
[795] no source work
[796] no source work
[798] no source work
[799] no source work
[804] no source work
[810] no source work
[811] no source work
[815] no source work
[816] no source work
[819] no sou

In [17]:
# =========================================================
# confirm_hdgb101 内の *_500ps_hdgb_work を全部リンク化する版
# IDごとにfindしないので、no source work問題を避ける
# =========================================================

SRC_BASE = "/home/teraimao/experiment/confirm_hdgb101"
LINK_BASE = "/home/teraimao/experiment/confirm_hdgb101_for_aggregation"

cmd = f"""
set -e

SRC_BASE="{SRC_BASE}"
LINK_BASE="{LINK_BASE}"

mkdir -p "$LINK_BASE"

echo "source base: $SRC_BASE"
echo "link base  : $LINK_BASE"
echo ""

echo "===== source work dirs count ====="
find "$SRC_BASE" -maxdepth 1 -type d -name "*_500ps_hdgb_work" | wc -l

echo ""
echo "===== source examples ====="
find "$SRC_BASE" -maxdepth 1 -type d -name "*_500ps_hdgb_work" | sort | head -n 10

echo ""
echo "===== making links ====="

linked=0

for orig in "$SRC_BASE"/*_500ps_hdgb_work; do
    if [ ! -d "$orig" ]; then
        continue
    fi

    base=$(basename "$orig")

    # 例:
    # 234_2026.06.30_500ps_hdgb_work
    # → 234_2026.06.30_500ps_work
    newbase="${{base/_500ps_hdgb_work/_500ps_work}}"

    link="$LINK_BASE/$newbase"

    mkdir -p "$link"

    for d in hdgb_1 hdgb_2 hdgb_3 hdgb_4 hdgb_5 analysis; do
        if [ -e "$orig/$d" ]; then
            ln -sfn "$orig/$d" "$link/$d"
        fi
    done

    linked=$((linked + 1))
done

echo ""
echo "linked works: $linked"

echo ""
echo "===== linked work count ====="
find "$LINK_BASE" -maxdepth 1 -type d -name "*_500ps_work" | wc -l

echo ""
echo "===== linked examples ====="
find "$LINK_BASE" -maxdepth 1 -type d -name "*_500ps_work" | sort | head -n 10

echo ""
echo "===== check ID234 example ====="
w=$(find "$LINK_BASE" -maxdepth 1 -type d -name "234_*_500ps_work" | head -n 1)
echo "work=$w"

if [ -n "$w" ]; then
    echo ""
    echo "inside:"
    ls -l "$w" | head -n 30

    echo ""
    test -d "$w/hdgb_1" && echo "OK: hdgb_1" || echo "NG: hdgb_1"
    test -d "$w/analysis/ana_hdgb_1" && echo "OK: analysis/ana_hdgb_1" || echo "NG: analysis/ana_hdgb_1"
fi
"""

stdin, stdout, stderr = client.exec_command(cmd)
print(stdout.read().decode())

err = stderr.read().decode()
if err:
    print("stderr:")
    print(err)

source base: /home/teraimao/experiment/confirm_hdgb101
link base  : /home/teraimao/experiment/confirm_hdgb101_for_aggregation

===== source work dirs count =====
793

===== source examples =====
/home/teraimao/experiment/confirm_hdgb101/1000_2026.07.03_500ps_hdgb_work
/home/teraimao/experiment/confirm_hdgb101/101_2026.06.30_500ps_hdgb_work
/home/teraimao/experiment/confirm_hdgb101/102_2026.06.30_500ps_hdgb_work
/home/teraimao/experiment/confirm_hdgb101/103_2026.06.30_500ps_hdgb_work
/home/teraimao/experiment/confirm_hdgb101/104_2026.06.30_500ps_hdgb_work
/home/teraimao/experiment/confirm_hdgb101/105_2026.06.30_500ps_hdgb_work
/home/teraimao/experiment/confirm_hdgb101/106_2026.06.30_500ps_hdgb_work
/home/teraimao/experiment/confirm_hdgb101/107_2026.06.30_500ps_hdgb_work
/home/teraimao/experiment/confirm_hdgb101/108_2026.06.30_500ps_hdgb_work
/home/teraimao/experiment/confirm_hdgb101/109_2026.06.30_500ps_hdgb_work

===== making links =====

linked works: 793

===== linked work count ====

In [18]:
# =========================================================
# HDGB 101~1000 をリンク用ディレクトリから再集計
# =========================================================

job_hdgb_101_1000 = submit_md_aggregation_background_custom_name(
    client=client,
    model="hdgb",
    base_dir="/home/teraimao/experiment/confirm_hdgb101_for_aggregation",
    start_id=101,
    end_id=1000,
    target_date="ALL",
    target_t_end_ps="500",
    skip_ps=250.0,
    dt_ps=1.0,
    tolerance_frames=0,
    save_csv_name="master_database_hdgb.csv",
)

current_job["job"] = job_hdgb_101_1000

print("✅ submitted")
print("model      :", job_hdgb_101_1000["model"])
print("remote_csv :", job_hdgb_101_1000["remote_csv"])
print("remote_log :", job_hdgb_101_1000["remote_log"])
print("save_csv   :", job_hdgb_101_1000["save_csv_name"])

===== MODEL/WORK STRUCTURE CHECK =====
model=$'hdgb'
base=/home/teraimao/experiment/confirm_hdgb101_for_aggregation
expected md dir: hdgb_1
expected analysis dir: analysis/ana_hdgb_1
found_work=/home/teraimao/experiment/confirm_hdgb101_for_aggregation/101_2026.06.30_500ps_work
__MD_DIR_OK__
__ANALYSIS_DIR_OK__

🚀 集計をバックグラウンド投入しました
model: hdgb
base_dir: /home/teraimao/experiment/confirm_hdgb101_for_aggregation
PID: 2696497
remote_csv: /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_191426.csv
remote_log: /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_191426.log
local_save_csv: master_database_hdgb.csv
✅ submitted
model      : hdgb
remote_csv : /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_191426.csv
remote_log : /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_191426.log
s

In [22]:
# =========================================================
# HDGB 101~1000 再集計の進捗確認
# =========================================================

job_hdgb_101_1000 = {
    "model": "hdgb",
    "pid": "2696497",
    "remote_csv": "/home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_191426.csv",
    "remote_log": "/home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_191426.log",
    "base_dir": "/home/teraimao/experiment/confirm_hdgb101_for_aggregation",
    "save_csv_name": "master_database_hdgb.csv",
}

current_job["job"] = job_hdgb_101_1000

check_md_aggregation_job(client, job_hdgb_101_1000)

===== PID CHECK =====
    PID STAT     ELAPSED CMD

===== CSV CHECK =====
-rw-rw-r-- 1 teraimao teraimao 1.6M Jul  6 19:26 /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_191426.csv

===== LOG CHECK =====
-rw-rw-r-- 1 teraimao teraimao 18K Jul  6 19:26 /home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_191426.log

===== LOG TAIL =====
[927] no work dirs
[928] done, works=1
[929] done, works=1
[930] no work dirs
[931] no work dirs
[932] done, works=1
[933] done, works=1
[934] done, works=1
[935] done, works=1
[936] done, works=1
[937] done, works=1
[938] done, works=1
[939] no work dirs
[940] no work dirs
[941] no work dirs
[942] no work dirs
[943] no work dirs
[944] no work dirs
[945] no work dirs
[946] no work dirs
[947] no work dirs
[948] no work dirs
[949] no work dirs
[950] no work dirs
[951] no work dirs
[952] no work dirs
[953] no work dirs
[954] no work dirs
[955] done, works

In [23]:
# =========================================================
# HDGB 101~1000 を master_database_hdgb.csv に直接統合
# =========================================================

import os
import pandas as pd

SAVE_DIR = "/Users/teraimao/experiment/data/解析結果/"
os.makedirs(SAVE_DIR, exist_ok=True)

remote_csv = "/home/teraimao/experiment/aggregation_results/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_191426.csv"

local_tmp = os.path.join(
    SAVE_DIR,
    os.path.basename(remote_csv)
)

local_master_csv = os.path.join(
    SAVE_DIR,
    "master_database_hdgb.csv"
)

# サーバーからCSVを取得
sftp = client.open_sftp()
try:
    sftp.get(remote_csv, local_tmp)
finally:
    sftp.close()

print("✅ downloaded:")
print(local_tmp)

new_df = pd.read_csv(local_tmp)

print("\nnew csv shape:", new_df.shape)

if "Model" in new_df.columns:
    print("\nnew csv Model counts:")
    print(new_df["Model"].value_counts(dropna=False))

    actual_models = set(
        new_df["Model"].dropna().astype(str).str.lower().unique()
    )

    if actual_models != {"hdgb"}:
        raise ValueError(
            f"このCSVはHDGBだけではありません。統合しない方が安全です。actual_models={actual_models}"
        )

if "Status" in new_df.columns:
    print("\nnew csv Status counts:")
    print(new_df["Status"].value_counts(dropna=False))

# 既存masterに統合
if os.path.exists(local_master_csv):
    old_df = pd.read_csv(local_master_csv)

    final_df = pd.concat(
        [old_df, new_df],
        ignore_index=True
    )

    final_df = final_df.drop_duplicates(
        subset=["Model", "ID", "Date", "Pos_Num", "t_end_ps"],
        keep="last"
    )
else:
    final_df = new_df.copy()

# 数値列を整える
for col in ["ID", "Pos_Num", "t_end_ps"]:
    if col in final_df.columns:
        final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

# 並び替え
sort_cols = [
    col for col in ["ID", "Date", "t_end_ps", "Pos_Num"]
    if col in final_df.columns
]

final_df = final_df.sort_values(sort_cols).reset_index(drop=True)

# 保存
final_df.to_csv(local_master_csv, index=False)

print("\n✅ merged master saved:")
print(local_master_csv)

print("\nmaster shape:", final_df.shape)

print("\nModel counts:")
print(final_df["Model"].value_counts(dropna=False))

print("\nStatus counts:")
print(final_df["Status"].value_counts(dropna=False))

print("\nID range:")
print(final_df["ID"].min(), "~", final_df["ID"].max())

print("\nID count:")
print(final_df["ID"].nunique())

display(final_df.tail(20))

✅ downloaded:
/Users/teraimao/experiment/data/解析結果/master_database_hdgb_hdgb_ID101_1000_ALL_500ps_20260706_191426.csv

new csv shape: (3965, 22)

new csv Model counts:
Model
hdgb    3965
Name: count, dtype: int64

new csv Status counts:
Status
Failed    3965
Name: count, dtype: int64

✅ merged master saved:
/Users/teraimao/experiment/data/解析結果/master_database_hdgb.csv

master shape: (4440, 22)

Model counts:
Model
hdgb    4440
Name: count, dtype: int64

Status counts:
Status
Failed     3965
Success     475
Name: count, dtype: int64

ID range:
1 ~ 1000

ID count:
888


,Model,ID,Date,t_end_ps,Cons_Label,Pos_Num,Z,WorkDir,LogPath,HB_Path,...,Message,skip_ps,dt_ps,tolerance_frames,Normal_Termination,HB_Avg,SASA_Avg,Solv_Free_Avg,vdW_Avg,Coul_Avg
4420,hdgb,997,2026.07.03,500.0,NaN,1,0,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,...,SASA parse failed,250.0,1.0,0,True,3.164,NaN,-4.29301,12.63513,61.70745
4421,hdgb,997,2026.07.03,500.0,NaN,2,7,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,...,SASA parse failed,250.0,1.0,0,True,2.696,NaN,-8.07824,12.88289,59.97354
4422,hdgb,997,2026.07.03,500.0,NaN,3,15,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,...,SASA parse failed,250.0,1.0,0,True,2.376,NaN,-18.16663,12.95490,62.19945
4423,hdgb,997,2026.07.03,500.0,NaN,4,17,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,...,SASA parse failed,250.0,1.0,0,True,2.164,NaN,-18.65222,12.82319,59.57630
4424,hdgb,997,2026.07.03,500.0,NaN,5,20,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,...,SASA parse failed,250.0,1.0,0,True,1.956,NaN,-22.41534,12.93771,62.21130
4425,hdgb,998,2026.07.03,500.0,NaN,1,0,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,...,SASA parse failed,250.0,1.0,0,True,2.724,NaN,-5.09474,12.79334,48.33715
4426,hdgb,998,2026.07.03,500.0,NaN,2,7,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,...,SASA parse failed,250.0,1.0,0,True,2.480,NaN,-8.53626,12.31905,49.63081
4427,hdgb,998,2026.07.03,500.0,NaN,3,15,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,...,SASA parse failed,250.0,1.0,0,True,2.472,NaN,-18.70592,11.41269,55.89459
4428,hdgb,998,2026.07.03,500.0,NaN,4,17,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,...,SASA parse failed,250.0,1.0,0,True,2.408,NaN,-23.65579,12.87865,56.19018
4429,hdgb,998,2026.07.03,500.0,NaN,5,20,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,/home/teraimao/experiment/confirm_hdgb101_for_...,...,SASA parse failed,250.0,1.0,0,True,2.508,NaN,-26.62355,12.84746,56.36865
